# Emotion Detection Model Training
### FER2013 Dataset — CNN Model

**Steps:**
1. Kaggle se dataset download karo
2. Model train karo
3. `emotion_model.keras` download karo
4. Apne project mein replace karo

> Runtime > Change runtime type > **T4 GPU** select karo pehle

## Step 1 — Kaggle Setup

In [ ]:
# Kaggle API install karo
!pip install kaggle -q

# Kaggle account pe jao → Account → API → Create New Token
# kaggle.json download hoga — usse upload karo
from google.colab import files
files.upload()  # kaggle.json upload karo

In [ ]:
import os
os.makedirs('/root/.kaggle', exist_ok=True)
!cp kaggle.json /root/.kaggle/
!chmod 600 /root/.kaggle/kaggle.json

# Dataset download karo
!kaggle datasets download -d msambare/fer2013
!unzip -q fer2013.zip -d fer2013
print('Dataset ready!')

## Step 2 — Dataset Check

In [ ]:
import os

train_dir = '/content/fer2013/train'
test_dir  = '/content/fer2013/test'

classes = os.listdir(train_dir)
print('Emotions found:', classes)

for c in classes:
    count = len(os.listdir(os.path.join(train_dir, c)))
    print(f'  {c}: {count} images')

## Step 3 — Data Preprocessing

In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator

IMG_SIZE = 48
BATCH_SIZE = 64

# Training data — augmentation ke saath
train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=15,
    width_shift_range=0.15,
    height_shift_range=0.15,
    horizontal_flip=True,
    zoom_range=0.15,
    fill_mode='nearest'
)

# Test data — sirf rescale
test_datagen = ImageDataGenerator(rescale=1./255)

train_gen = train_datagen.flow_from_directory(
    train_dir,
    target_size=(IMG_SIZE, IMG_SIZE),
    color_mode='grayscale',
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=True
)

test_gen = test_datagen.flow_from_directory(
    test_dir,
    target_size=(IMG_SIZE, IMG_SIZE),
    color_mode='grayscale',
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=False
)

print('Class indices:', train_gen.class_indices)
print('Train samples:', train_gen.samples)
print('Test samples:', test_gen.samples)

## Step 4 — CNN Model

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (
    Conv2D, MaxPooling2D, BatchNormalization,
    Dropout, Flatten, Dense
)
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import (
    EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
)

num_classes = len(train_gen.class_indices)

model = Sequential([
    # Block 1
    Conv2D(64, (3,3), padding='same', activation='relu', input_shape=(48,48,1)),
    BatchNormalization(),
    Conv2D(64, (3,3), padding='same', activation='relu'),
    BatchNormalization(),
    MaxPooling2D(2,2),
    Dropout(0.25),

    # Block 2
    Conv2D(128, (3,3), padding='same', activation='relu'),
    BatchNormalization(),
    Conv2D(128, (3,3), padding='same', activation='relu'),
    BatchNormalization(),
    MaxPooling2D(2,2),
    Dropout(0.25),

    # Block 3
    Conv2D(256, (3,3), padding='same', activation='relu'),
    BatchNormalization(),
    Conv2D(256, (3,3), padding='same', activation='relu'),
    BatchNormalization(),
    MaxPooling2D(2,2),
    Dropout(0.25),

    # Fully Connected
    Flatten(),
    Dense(512, activation='relu'),
    BatchNormalization(),
    Dropout(0.5),
    Dense(256, activation='relu'),
    BatchNormalization(),
    Dropout(0.5),
    Dense(num_classes, activation='softmax')
])

model.compile(
    optimizer=Adam(learning_rate=0.001),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()

## Step 5 — Train

In [ ]:
callbacks = [
    # Validation loss improve nahi hui toh LR kam karo
    ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=4,
        min_lr=1e-6,
        verbose=1
    ),
    # Best model save karo
    ModelCheckpoint(
        'best_model.keras',
        monitor='val_accuracy',
        save_best_only=True,
        verbose=1
    ),
    # Zyada improve nahi ho raha toh band karo
    EarlyStopping(
        monitor='val_accuracy',
        patience=10,
        restore_best_weights=True,
        verbose=1
    )
]

history = model.fit(
    train_gen,
    validation_data=test_gen,
    epochs=60,
    callbacks=callbacks
)

## Step 6 — Results

In [ ]:
import matplotlib.pyplot as plt

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(history.history['accuracy'], label='Train')
ax1.plot(history.history['val_accuracy'], label='Validation')
ax1.set_title('Accuracy')
ax1.legend()

ax2.plot(history.history['loss'], label='Train')
ax2.plot(history.history['val_loss'], label='Validation')
ax2.set_title('Loss')
ax2.legend()

plt.tight_layout()
plt.show()

# Final accuracy
loss, acc = model.evaluate(test_gen)
print(f'\nTest Accuracy: {acc*100:.2f}%')

## Step 7 — Save & Download

In [ ]:
# Best model already best_model.keras mein save ho gaya hai
# Rename karke download karo
import shutil
shutil.copy('best_model.keras', 'emotion_model.keras')

# Download
from google.colab import files
files.download('emotion_model.keras')
print('Model download ho gaya!')

## Step 8 — Class Order Check (Important!)

In [ ]:
# Yeh order apne main.py ke emotion_labels se match hona chahiye
class_indices = train_gen.class_indices
labels = {v: k for k, v in class_indices.items()}
print('Model ka emotion order:')
for i in sorted(labels.keys()):
    print(f'  {i}: {labels[i]}')

print('\nApne main.py mein yeh order use karo:')
ordered = [labels[i] for i in sorted(labels.keys())]
print('emotion_labels =', ordered)